###### Author: Basit Rohan
###### Date: Octobar 2025
###### Tools: Numpy, Pandas

In [2]:
import numpy as np
import pandas as pd

In [3]:
df_ci = pd.read_csv('client_inquiries_simulated.csv')
df_ci

,inquiry_id,client_id,created_at,resolved_at,resolution_minutes,issue_category,issue_subcategory,priority,sla_hours,resolved_within_sla,satisfaction_score,channel,region,handled_by
0,INC00001,C1758,2/27/2025 0:47,2/27/2025 15:09,862.0,Fraud Detection,Card Fraud,Medium,48,True,4.0,Phone,Central,Agent_55
1,INC00002,C1733,1/17/2025 0:05,1/17/2025 13:52,827.0,Fraud Detection,Suspicious Login,Low,72,True,NaN,Mobile App,South,Agent_58
2,INC00003,C1220,5/23/2025 0:48,5/25/2025 3:43,3055.0,Delay,Interbank Delay,Low,72,True,NaN,Chat,North,Agent_12
3,INC00004,C1470,7/14/2025 3:22,7/14/2025 13:53,631.0,Transaction Mismatch,Duplicate Transaction,High,24,True,4.0,Phone,West,Agent_11
4,INC00005,C1046,5/31/2025 20:39,6/1/2025 23:35,1616.0,KYC/Documentation,Mismatch Info,Low,72,True,4.0,Email,East,Agent_11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,INC04996,C1046,6/29/2025 0:29,6/30/2025 15:39,2350.0,KYC/Documentation,Missing Documents,Medium,48,True,2.0,Email,Central,Agent_46
4996,INC04997,C1887,1/31/2025 20:45,2/2/2025 5:25,1960.0,Delay,Processing Queue,Medium,48,True,NaN,Chat,East,Agent_1
4997,INC04998,C1392,1/18/2025 10:12,1/18/2025 23:55,823.0,Transaction Mismatch,Missing Transaction,Medium,48,True,4.0,Phone,South,Agent_51
4998,INC04999,C1287,8/20/2025 6:13,8/20/2025 23:41,1048.0,Delay,Clearing Delay,Critical,4,False,1.0,Phone,East,Agent_98


In [4]:
df_ci.isna().sum()

inquiry_id               0
client_id                0
created_at               0
resolved_at            245
resolution_minutes     245
issue_category           0
issue_subcategory        0
priority                 0
sla_hours                0
resolved_within_sla      0
satisfaction_score     613
channel                  0
region                   0
handled_by               0
dtype: int64

In [5]:
df_ci.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   inquiry_id           5000 non-null   object 
 1   client_id            5000 non-null   object 
 2   created_at           5000 non-null   object 
 3   resolved_at          4755 non-null   object 
 4   resolution_minutes   4755 non-null   float64
 5   issue_category       5000 non-null   object 
 6   issue_subcategory    5000 non-null   object 
 7   priority             5000 non-null   object 
 8   sla_hours            5000 non-null   int64  
 9   resolved_within_sla  5000 non-null   bool   
 10  satisfaction_score   4387 non-null   float64
 11  channel              5000 non-null   object 
 12  region               5000 non-null   object 
 13  handled_by           5000 non-null   object 
dtypes: bool(1), float64(2), int64(1), object(10)
memory usage: 512.8+ KB


In [6]:
df_ci.describe()

,resolution_minutes,sla_hours,satisfaction_score
count,4755.000000,5000.000000,4387.000000
mean,936.327234,50.624000,3.961705
std,935.301561,20.972043,1.250587
min,-37.000000,4.000000,1.000000
25%,296.000000,24.000000,3.000000
50%,601.000000,48.000000,4.000000
75%,1238.000000,72.000000,5.000000
max,6438.000000,72.000000,5.000000


In [ ]:
# since there are na values in 3 columns so we need to deal with them first
df_ci['resolved_at'].fillna('Unresolved', inplace= True)
# leaving resolution_minutes as nan  
# also keep the satisfaction score as nan since missing values could be because of 
# customer didnt respond to the survey or tickets werent closed with survey

Now making the tables for the database

In [ ]:
#clients
clients = df_ci[['client_id']].drop_duplicates()
clients.to_csv("client.csv",index = False)

In [22]:
#regions
regions = df_ci[['region']].drop_duplicates().reset_index(drop=True)
regions['region_id'] = regions.index + 1
regions = regions[['region_id','region']]
regions.to_csv('regions.csv',index=False)

In [23]:
#priorities
priorities = df_ci[['priority','sla_hours']].drop_duplicates().reset_index(drop=True)
priorities['priority_id'] = priorities.index + 1
priorities = priorities[['priority_id','priority','sla_hours']]
priorities.to_csv('priorities.csv', index = False)


In [24]:
#agents
agents = df_ci[['handled_by']].drop_duplicates().reset_index(drop=True)
agents['agent_id'] = agents.index + 1
agents = agents[['agent_id','handled_by']]
agents.to_csv('agents.csv',index = False)

In [25]:
#channels
channels = df_ci[['channel']].drop_duplicates().reset_index(drop=False)
channels['channel_id'] = channels.index + 1
channels = channels[['channel_id','channel']]
channels.to_csv('channels.csv', index = False)

In [27]:
#issue_category
issue_categories = df_ci[['issue_category']].drop_duplicates().reset_index(drop=True)
issue_categories['issue_category_id'] = issue_categories.index + 1
issue_categories = issue_categories[['issue_category_id','issue_category']]
issue_categories.to_csv('issue_categories.csv', index = False)

In [ ]:
#issue_subcategory
#category + subcategory
issue_subcategories = df_ci[['issue_category', 'issue_subcategory']].drop_duplicates().reset_index(drop=True)
issue_subcategories['issue_category_id'] = issue_subcategories.groupby('issue_category').ngroup() + 1
issue_subcategories['issue_subcategory_id'] = issue_subcategories.index + 1
issue_subcategories = issue_subcategories[['issue_subcategory_id', 'issue_category_id', 'issue_subcategory']]
issue_subcategories.to_csv("issue_subcategories.csv", index=False)


In [39]:
#loading the other tables for FK and facts
agents = pd.read_csv("agents.csv")
issue_categories = pd.read_csv("issue_categories.csv")
issue_subcategories = pd.read_csv("issue_subcategories.csv")
priorities = pd.read_csv("priorities.csv")
channels = pd.read_csv("channels.csv")
regions = pd.read_csv("regions.csv")

# Merge to replace text with IDs
df_ci = df_ci.merge(agents[['handled_by']], on="handled_by", how="left")
df_ci = df_ci.merge(issue_categories[['issue_category']], on="issue_category", how="left")
df_ci = df_ci.merge(issue_subcategories[['issue_subcategory','issue_subcategory_id']], on="issue_subcategory", how="left")
df_ci = df_ci.merge(priorities[['priority','priority_id']], on="priority", how="left")
df_ci = df_ci.merge(channels[['channel','channel_id']], on="channel", how="left")
df_ci = df_ci.merge(regions[['region','region_id']], on="region", how="left")

# Build fact table
inquiries = df_ci[['inquiry_id', 'client_id', 'issue_category_id', 'issue_subcategory_id',
                   'priority_id', 'channel_id', 'region_id', 'agent_id',
                   'created_at', 'resolved_at', 'resolution_minutes',
                   'resolved_within_sla', 'satisfaction_score']]

inquiries.to_csv("inquiries.csv", index=False)
